# Experimentos para mejorar el modelo
**Problema:** la misma receta dio 0,69 y 0,60 de exactitud en prueba (variación por semilla), y las fotos de 1920×1080 se reducen a 224×224, donde las manchas pequeñas de la roya casi desaparecen.

**Qué hace este notebook**
1. Entrena la receta ganadora (4b) con varios **tamaños de entrada** (224, 320, 448 px) y **3 semillas** cada uno.
2. Compara usando **solo validación** (media ± desviación y un *conjunto* de las 3 semillas). El test no se toca hasta el final.
3. Con el mejor tamaño arma un **conjunto de modelos** (promedia las 3 semillas), lo evalúa **una sola vez** en test y lo exporta a ONNX para la web.

**Funciona en Kaggle, Colab, PyCharm y Jupyter.** Es **reanudable**: cada corrida terminada se guarda; si se corta, vuelva a ejecutar y salta lo ya entrenado.
- **Kaggle:** Settings → Accelerator **GPU**, Internet **On**; use *Save Version → Save & Run All* y puede cerrar el navegador. Se detiene solo a las 10 h con todo guardado; para continuar, agregue el *Output* de esa versión como Input y vuelva a ejecutar.
- **Colab:** GPU T4; con `USAR_DRIVE = True` los modelos parciales quedan en su Google Drive.
- **Sin GPU el entrenamiento completo puede tardar muchas horas o días.**

> No es seguro que mejore: el resultado puede ser que 224 px sea suficiente.

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

EN_COLAB = os.path.exists("/content")
EN_KAGGLE = os.path.exists("/kaggle/working")
USAR_DRIVE = True                       # solo Colab: guarda los modelos parciales en tu Google Drive
LIMITE_HORAS = 10 if EN_KAGGLE else None  # Kaggle corta las sesiones a las 12 h; aquí se detiene solo antes, con todo guardado
URL_REPO = "https://github.com/JavierAnd0/ia-Kaggle-Coffee"

def clonar(destino):
    if not os.path.isdir(destino):
        r = subprocess.run(["git", "clone", "-q", "-b", "main", URL_REPO, destino])
        if r.returncode != 0:
            raise SystemExit("No pude clonar el repositorio. En Kaggle active Internet: panel derecho > Settings > Internet > On "
                             "(pide verificar el teléfono la primera vez).")

if EN_COLAB:
    clonar("/content/ia-Kaggle-Coffee")
    os.chdir("/content/ia-Kaggle-Coffee")
elif EN_KAGGLE:
    clonar("/kaggle/working/ia-Kaggle-Coffee")
    os.chdir("/kaggle/working/ia-Kaggle-Coffee")
    os.environ.setdefault("CHECKPOINT_DIR", "/kaggle/working/checkpoints")
    # Continuar una ejecución anterior: agregue su "Output" como Input (Add Input > Your Work > Notebook Output).
    nuevos = 0
    for previo in list(Path("/kaggle/input").glob("*/checkpoints")) + list(Path("/kaggle/input").glob("*/*/checkpoints")):
        destino = Path(os.environ["CHECKPOINT_DIR"]); destino.mkdir(parents=True, exist_ok=True)
        for f in previo.iterdir():
            if not (destino / f.name).exists():
                shutil.copy2(f, destino / f.name); nuevos += 1
    if nuevos:
        print(f"Recuperados {nuevos} archivos de una ejecución anterior.")

# Subir hasta la carpeta del proyecto (la que tiene src/ y data_splits/), desde donde se abra el notebook.
raiz = Path.cwd()
while not ((raiz / "src").is_dir() and (raiz / "data_splits").is_dir()) and raiz != raiz.parent:
    raiz = raiz.parent
if not (raiz / "src").is_dir():
    raise SystemExit("No encuentro la carpeta del proyecto (con src/ y data_splits/). Abra el notebook dentro del repositorio.")
os.chdir(raiz)
sys.path.insert(0, str(raiz))
print("Proyecto:", raiz)

# Instala solo lo que falte
faltan = []
for modulo, paquete in (("keras_tuner", "keras-tuner"), ("onnx", "onnx"), ("tf2onnx", "tf2onnx"), ("onnxruntime", "onnxruntime"),
                        ("sklearn", "scikit-learn"), ("pandas", "pandas"), ("tensorflow", "tensorflow")):
    try:
        __import__(modulo)
    except ImportError:
        faltan.append(paquete)
if faltan:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltan], check=True)

import json, numpy as np, pandas as pd, tensorflow as tf
from src import config, generators, tuning, experiments, export

DIR_MODELOS = None
if EN_COLAB and USAR_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DIR_MODELOS = "/content/drive/MyDrive/cafe_checkpoints"
        print("Modelos parciales en Google Drive:", DIR_MODELOS)
    except Exception as e:
        print("No se pudo montar Drive; se guardará en el disco temporal de Colab:", e)

FAST_MODE = False                 # True = prueba rápida (pocas épocas, 2 tamaños, 2 semillas); sus resultados no sirven para decidir
TAMANOS = (224, 320, 448) if not FAST_MODE else (224, 320)
SEMILLAS = (0, 1, 2) if not FAST_MODE else (0, 1)
EPOCAS, EPOCAS_FT = (30, 10) if not FAST_MODE else (2, 1)
TAMANO_FINAL = None               # None = elegir el mejor por validación; o fijar uno (p. ej. 320)
tf.keras.utils.set_random_seed(config.SEED)
print("GPU:", tf.config.list_physical_devices("GPU") or "NINGUNA (será muy lento)")
print("Carpeta de modelos parciales:", experiments.carpeta_checkpoints(DIR_MODELOS))

In [ ]:
tr = pd.read_csv(config.SPLITS_DIR / "train.csv")
va = pd.read_csv(config.SPLITS_DIR / "val.csv")
te = pd.read_csv(config.SPLITS_DIR / "test.csv")
li = generators.make_label_index(tr["label"])
hp = tuning.load_best_hp()
print(li, len(tr), len(va), len(te)); hp.values

## 1. Rejilla: tamaño × semilla (solo validación)
Cada línea impresa es un modelo terminado. Si se corta, vuelva a ejecutar esta celda: las corridas ya hechas se reutilizan.

In [ ]:
PARCIAL = False
try:
    corridas, resumen = experiments.correr_rejilla(
        hp, tr, va, li, tamanos=TAMANOS, semillas=SEMILLAS, epocas=EPOCAS, epocas_ft=EPOCAS_FT, dir_modelos=DIR_MODELOS,
        limite_segundos=LIMITE_HORAS * 3600 if LIMITE_HORAS else None)
except experiments.TiempoAgotado as e:
    PARCIAL = True
    print(e)
    print("Todo lo entrenado quedó guardado. En Kaggle: guarde esta versión, agregue su Output como Input de un nuevo "
          "Save & Run All y vuelva a ejecutar; continuará donde quedó.")
if not PARCIAL:
    config.RESULTS_DIR.mkdir(exist_ok=True)
    corridas.to_csv(config.RESULTS_DIR / "experimentos_corridas.csv", index=False)
    resumen.to_csv(config.RESULTS_DIR / "experimentos_resumen.csv")
    print(resumen.round(3).to_string())

**Cómo leer la tabla:** `f1_media ± f1_desv` es el F1 macro de las corridas sueltas; `f1_conjunto` es el del promedio de las semillas.
Si la diferencia entre tamaños es menor que la desviación, **no hay evidencia de que un tamaño sea mejor**: en ese caso conviene elegir el más pequeño (más rápido en la web).

In [ ]:
if PARCIAL:
    print("Ejecución parcial: se omite la elección y la exportación hasta completar la rejilla.")
else:
    mejor = int(resumen["f1_conjunto"].idxmax()) if TAMANO_FINAL is None else TAMANO_FINAL
    print("Tamaño elegido:", mejor)
    print(resumen.loc[[mejor]].round(3).to_string())

## 2. Conjunto de modelos del tamaño elegido: test (una sola vez) y exportación para la web
Genera `web/model/` (ONNX + etiquetas + ficha + paridad) y `metricas_web.json`, que la web usa para mostrar cifras **del modelo exportado**.

In [ ]:
if not PARCIAL:
    from sklearn.metrics import classification_report, confusion_matrix
    r = experiments.finalizar(hp, tr, va, te, li, resumen, SEMILLAS, DIR_MODELOS, config.ROOT / "web" / "model", TAMANO_FINAL)
    nombres = sorted(li, key=li.get)
    print("Versión:", r["version"], "| diferencia Keras vs ONNX:", r["diferencia_onnx"])
    print("VALIDACIÓN:", r["metricas"]["val"])
    print("TEST      :", r["metricas"]["test"])
    pred = r["p_test"].argmax(1)
    print(classification_report(r["y_test"], pred, target_names=nombres, zero_division=0))
    print(pd.DataFrame(confusion_matrix(r["y_test"], pred), index=nombres, columns=nombres))

In [ ]:
if not PARCIAL:
    zip_path = export.empaquetar(config.ROOT / "modelo_cafe.zip", [config.ROOT / "web" / "model"],
                                 [config.RESULTS_DIR / "experimentos_corridas.csv", config.RESULTS_DIR / "experimentos_resumen.csv"])
    if EN_KAGGLE:                      # lo que está en /kaggle/working aparece en la pestaña Output
        shutil.copy2(zip_path, "/kaggle/working/modelo_cafe.zip")
        print("Paquete listo: pestaña Output de esta versión > modelo_cafe.zip")
    else:
        print("Paquete listo:", zip_path)
    try:                               # descarga automática solo en Colab
        from google.colab import files
        files.download(str(zip_path))
    except ImportError:
        pass

## 3. Qué hacer con el resultado
- Descargue `modelo_cafe.zip` y copie `web/model/*` a `web/model/` del repositorio; súbalo. La web lo carga sola y muestra las cifras de `metricas_web.json`.
- Compare `f1_conjunto` con el modelo actual (validación: exactitud 0,73, F1 macro 0,68). Si **no mejora**, no lo reemplace: quédese con el actual.
- Pendiente (no se resuelve aquí): validar con café arábica (BRACOL o JMuBEN) y con fotos de cultivos colombianos.